# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Refresh / Content Opportunity Scoring**. I will frame it mainly as a **ranking/scoring** task, because the real decision is "which pages should a reviewer inspect first?" rather than "is this page good or bad?" The output should be a prioritized review queue of anonymized content pages, with higher scores for pages that look more worth human review.

A classification proxy can support the ranking. For example, I can estimate whether a page appears to be declining, then combine that signal with visibility, CTR, freshness, position, and engagement. But the user-facing output should still be a ranked/scored list, not an automatic yes/no decision.


In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
data_path = next(path for path in candidates if path.exists())
raw_df = pd.read_csv(data_path)

lane_columns = [
    'content_id',
    'client_id',
    'content_type',
    'main_intent',
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'content_age_days',
    'days_since_last_update',
    'engagement_rate',
    'scroll_rate',
    'trend_direction',
    'trend_pct',
]

# Starter-pipeline eligibility for refresh work: keep pages with search visibility and enough age.
refresh_df = raw_df.loc[
    (raw_df['impressions_90d'] > 0) & (raw_df['content_age_days'] >= 90),
    lane_columns,
].copy()

pd.DataFrame({
    'framing_piece': ['task_type', 'lane_slice_rows', 'unit_of_analysis'],
    'my_choice': [
        'ranking/scoring, supported by a classification proxy',
        len(refresh_df),
        'one row = one pseudonymized content page/content item',
    ],
})


,framing_piece,my_choice
0,task_type,"ranking/scoring, supported by a classification..."
1,lane_slice_rows,30000
2,unit_of_analysis,one row = one pseudonymized content page/conte...


## 2. Target or proxy

My starter target proxy is **`is_declining_proxy`**, defined as:

```text
is_declining_proxy = trend_direction == "down"
```

This is a **proxy**, not a perfect target. It comes from the starter dataset's current trend bucket, so it helps me practice the workflow but does not prove future decline. Because this proxy is derived from `trend_direction` and related trend calculations, I should not use `trend_direction` or `trend_pct` as ordinary model features when training against this proxy.

A stronger capstone target would be future-looking, such as "features from a prior 90-day window -> decline or recovery in the next 30 days." For Week 2, this proxy is enough to sketch the ML task honestly.


In [2]:
target_df = refresh_df.copy()
target_df['is_declining_proxy'] = target_df['trend_direction'].eq('down').astype(int)

target_summary = pd.DataFrame({
    'target_value': [0, 1],
    'meaning': ['not currently in down trend bucket', 'currently in down trend bucket'],
    'row_count': [
        int((target_df['is_declining_proxy'] == 0).sum()),
        int((target_df['is_declining_proxy'] == 1).sum()),
    ],
})
target_summary['share_of_lane_slice'] = (
    target_summary['row_count'] / len(target_df)
).round(3)

target_summary


,target_value,meaning,row_count,share_of_lane_slice
0,0,not currently in down trend bucket,13738,0.458
1,1,currently in down trend bucket,16262,0.542


## 3. Success metric

My main success metric is **Precision@50**. This matches the real action: if a reviewer can only inspect 50 pages, I care about how many pages in the top 50 are actually useful review candidates by the chosen proxy or later observed target.

In plain words:

```text
Precision@50 = useful candidates in the top 50 recommendations / 50
```

A good first goal is to beat a simple fixed-rule baseline, such as prioritizing visible stale pages or declining pages with demand. I will not use generic accuracy as the main metric because the review workflow happens at the top of a ranked queue, not across every page equally.


In [3]:
review_capacity = 50
base_rate = target_df['is_declining_proxy'].mean()
expected_random_hits = review_capacity * base_rate

pd.DataFrame({
    'metric': ['Precision@50', 'base_rate_for_proxy', 'random_expected_proxy_hits_at_50'],
    'value': [
        'top_50_proxy_positives / 50',
        round(base_rate, 3),
        round(expected_random_hits, 1),
    ],
    'why_this_matters': [
        'matches a limited human review queue',
        'shows how common the proxy positive class is before modeling',
        'rough comparison point before building a real ranking',
    ],
})


,metric,value,why_this_matters
0,Precision@50,top_50_proxy_positives / 50,matches a limited human review queue
1,base_rate_for_proxy,0.542,shows how common the proxy positive class is b...
2,random_expected_proxy_hits_at_50,27.1,rough comparison point before building a real ...


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page/content item**. Each row has safe page-level signals from the starter data: visibility, traffic, CTR, average position, freshness, engagement, and the starter trend fields.

The dataframe below is my lane slice. It is not showing real URLs, client names, queries, or titles. The IDs are pseudonymized and should be used for grouping or traceability only, not as ML features.


In [4]:
unit_demo = target_df[[
    'content_id',
    'client_id',
    'content_type',
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'content_age_days',
    'days_since_last_update',
    'engagement_rate',
    'scroll_rate',
    'is_declining_proxy',
]].head(8)

unit_demo


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,engagement_rate,scroll_rate,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,0.76,10.6,187,20,5.88,4.55,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,0.05,20.3,445,25,0.00,10.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,0.09,36.5,141,20,0.00,28.57,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,0.49,6.2,463,22,1.28,3.45,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,0.13,44.0,263,14,0.00,24.29,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,5,0.03,8.5,147,20,0.00,25.00,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,1,0.00,7.0,90,20,0.00,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,28,0.06,21.2,445,22,3.57,7.14,0


## 5. Why ML beats a fixed rule here

A fixed rule like "refresh every old page" is too blunt. Some old pages may still be performing well, and some newer pages may already have weak CTR, weak engagement, or declining demand. A reviewer needs to balance several signals at once: impressions, clicks, CTR, average position, sessions, age, last update date, engagement, scroll behavior, and trend.

ML may beat a fixed rule because it can learn combinations of signals that are hard to write by hand. For example, a page with high impressions, page-one position, low CTR, and declining sessions may deserve a different action than a stale page with almost no search demand. The model still needs a transparent baseline, leakage checks, client-aware validation, and reason codes, because the output should support human review instead of hiding the decision inside a black box.


In [5]:
simple_rule = (
    target_df['trend_direction'].eq('down')
    & (target_df['impressions_90d'] >= 100)
)
low_ctr_visible = (
    (target_df['impressions_90d'] >= 500)
    & (target_df['avg_position'] > 0)
    & (target_df['avg_position'] <= 20)
    & (target_df['ctr'] < 0.5)
)
stale_visible = (
    (target_df['days_since_last_update'] >= 180)
    & (target_df['impressions_90d'] >= 500)
)

pd.DataFrame({
    'candidate_signal': [
        'declining_with_demand',
        'low_ctr_visible',
        'stale_visible',
    ],
    'rule_definition': [
        'trend_direction == down and impressions_90d >= 100',
        'impressions_90d >= 500, avg_position 1-20, ctr < 0.5%',
        'days_since_last_update >= 180 and impressions_90d >= 500',
    ],
    'matching_rows': [
        int(simple_rule.sum()),
        int(low_ctr_visible.sum()),
        int(stale_visible.sum()),
    ],
})


,candidate_signal,rule_definition,matching_rows
0,declining_with_demand,trend_direction == down and impressions_90d >=...,13152
1,low_ctr_visible,"impressions_90d >= 500, avg_position 1-20, ctr...",9759
2,stale_visible,days_since_last_update >= 180 and impressions_...,17


## Self-check

Before submitting, I checked each line honestly:

- [x] Every section above is filled with markdown thinking and code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are included.
- [x] My claims use careful words: proxy, observed, measured, directional, decision-support.
- [x] The work lives under `work/notebooks/`; after committing and pushing, I can submit my repo URL on the card.
